Script for comparing different CNN models 

In [ ]:
import os.path as op
import mne 
import os
from termcolor import colored
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,mean_absolute_error,accuracy_score,ConfusionMatrixDisplay, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib
import pickle
import tensorflow as tf

from sklearn.datasets import make_multilabel_classification
from sklearn.preprocessing import MultiLabelBinarizer

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential, Model  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, Conv1DTranspose, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2, l1

mne.set_log_level("CRITICAL")

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))

Loading in Data

In [ ]:
features_all = pd.read_pickle("training_features_18042026.pkl")
features_all = features_all.rename(columns={'Nap Number': 'Nap_ID'}) # making column names match 

trial_info = pd.read_csv('trial_info_duration_2904.csv')

# add duration information 
features_all = features_all.merge(
    trial_info[['Subject', 'Nap_ID', 'Triggers_Order_Nap', 'Duration_1', 'Duration_2']],
    on=['Subject', 'Nap_ID', 'Triggers_Order_Nap'],
    how='left'
)

features_all = features_all.rename(columns={
    'Duration_1': 'Duration_corr',
    'Duration_2': 'Duration_zygo'
})

features_all_store = features_all

x = np.isnan(features_all['Duration_zygo']) 
indices = np.where(x)[0]
features_all = features_all.drop(indices)

In [ ]:
np.shape(features_all_store)

## Functions

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [ ]:
# single head CNN model for duration 
def CNN_model_duration(input_shape, num_classes,feature_num):
    global epoch_len

    model = Sequential([
        Input(shape=(epoch_len, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu')) # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(2, activation='linear'))


    return model  # Return the compiled model

In [ ]:
def CNN_model_twohead(input_shape, num_classes,feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # Convolutional Layers - 
    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    x = Flatten()(x)  # Flatten the output of the convolutional layers


 
    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # Count-specific hidden layers
    count_branch = Dense(32, activation='relu')(shared)
    count_branch = Dropout(0.2)(count_branch)

    count_output = Dense(
        num_classes,
        activation='softmax',
        name='count_output'
    )(count_branch)

    duration_branch = Dense(32, activation='relu')(shared)
    duration_branch = Dropout(0.2)(duration_branch) # try increasing drop out for more regularization? (increase if overfitting)

    duration_output = Dense(
        num_classes,
        activation='linear',
        name='duration_output'
    )(duration_branch)


    model = Model(inputs=inputs, outputs=[count_output, duration_output])

    return model  # Return the compiled model

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_contraction_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(num_classes, activation='softmax', name="zygo_output")(x)

    # output head for Corr
    out_corr = Dense(num_classes, activation='softmax', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [ ]:
# single head CNN model for number of contractions  
def CNN_model_duration_multichan(input_shape, num_classes,feature_num):
    global epoch_len
    
    inputs = Input(shape=(epoch_len, feature_num))

    x = Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape)(inputs) # Add a 1D convolutional layer with 32 filters and ReLU activation
    x = MaxPooling1D(pool_size=2)(x) # Add a max pooling layer
    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)


    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)

    x = Flatten()(x)

    # two heads for each label 
    # output head for Zygo
    out_zygo = Dense(1, activation='relu', name="zygo_output")(x)

    # output head for Corr
    out_corr = Dense(1, activation='relu', name="corr_output")(x)

    model = Model(inputs=inputs, outputs=[out_zygo, out_corr])
    
    return model 

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, cvScores, model_name=None):
    # ---- Cross-validation stats ----
    avgScores = np.mean(cvScores)
    stdScores = np.std(cvScores)

    if model_name:
        print(f"\n===== {model_name} =====")

    print(f"Average KFold CV Score: {avgScores:.4f}")
    print(f"Std KFold CV Score: {stdScores:.4f}")

    # ---- Predictions ----
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Handle two-head model (take first output = count head)
    if isinstance(y_pred_train, list):
        y_pred_train = y_pred_train[0]
        y_pred_test = y_pred_test[0]

    # Convert probabilities → class labels
    y_pred_train = np.argmax(y_pred_train, axis=1)
    y_pred_test = np.argmax(y_pred_test, axis=1)

    # ---- Metrics ----
    accuracy_training = accuracy_score(y_train, y_pred_train)
    accuracy_test = accuracy_score(y_test, y_pred_test)

    f1_training = f1_score(y_train, y_pred_train, average='weighted')
    f1_test = f1_score(y_test, y_pred_test, average='weighted')

    # ---- Print results ----
    print(f"Training Accuracy: {accuracy_training:.4f}")
    print(f"Test Accuracy: {accuracy_test:.4f}")
    print(f"Training F1 Score: {f1_training:.4f}")
    print(f"Test F1 Score: {f1_test:.4f}")

    # ---- Return results (useful for logging/comparison) ----
    return {
        "cv_mean": avgScores,
        "cv_std": stdScores,
        "train_acc": accuracy_training,
        "test_acc": accuracy_test,
        "train_f1": f1_training,
        "test_f1": f1_test
    }

In [85]:
def CNN_model_fourhead_multichan(input_shape, num_classes, feature_num):
    global epoch_len

    inputs = Input(shape=(epoch_len, feature_num))

    # --- Shared CNN backbone ---
    x = Conv1D(32, kernel_size=3, activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    x = Conv1D(64, kernel_size=3, activation='relu')(x)
    x = MaxPooling1D(pool_size=1)(x)

    x = Flatten()(x)

    shared = Dense(128, activation='relu')(x)
    shared = Dropout(0.5)(shared)

    # zygo branch 
    zygo_branch = Dense(32, activation='relu')(shared)
    zygo_branch = Dropout(0.3)(zygo_branch)

    zygo_count_output = Dense(
        num_classes,
        activation='softmax',
        name='zygo_count_output'
    )(zygo_branch)

    zygo_duration_output = Dense(
        1,
        activation='linear',
        name='zygo_duration_output'
    )(zygo_branch)

    # corr branch 
    corr_branch = Dense(32, activation='relu')(shared)
    corr_branch = Dropout(0.2)(corr_branch)

    corr_count_output = Dense(
        num_classes,
        activation='softmax',
        name='corr_count_output'
    )(corr_branch)

    corr_duration_output = Dense(
        1,
        activation='linear',
        name='corr_duration_output'
    )(corr_branch)

    # --- Model ---
    model = Model(
        inputs=inputs,
        outputs=[
            zygo_count_output,
            zygo_duration_output,
            corr_count_output,
            corr_duration_output
        ]
    )

    return model

## Single Channel Training 

Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_contraction=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_contraction = CNN_model_contraction(input_shape, num_classes,feature_num)
    model_contraction.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_contraction.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_contraction.evaluate(X_test,y_test)
    cvScores_contraction.append(scores[1] * 100)

    k += 1 
    

model_history_contraction = model_contraction.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores_contraction)
stdScores = np.std(cvScores_contraction)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

# full training results (test data not seen during cross val)
y_pred_train = model_contraction.predict(X_train_full)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model_contraction.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train_full, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train_full, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test) 

Single Head Duration

In [ ]:
# Define CNN model inputs
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       


indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_duration=[]
epoch_num = 10 
k = 1
 
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]


    model_duration = CNN_model_duration(input_shape, num_classes,feature_num)
    model_duration.compile(optimizer='adam', loss='mae', metrics=['mae'])  

    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_duration.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_duration.evaluate(X_test,y_test)
    cvScores_duration.append(scores[1])

    k += 1 
    

model_history_duration = model_duration.fit(X_train_full, y_train_full, epochs=epoch_num, validation_data=(X_test, y_test)) #, callbacks=[early_stop])

In [ ]:
np.shape(y_train_full) 
np.shape(y) 

In [ ]:
# cross validation results 
cvScores_duration = np.array(cvScores_duration)
cvScores_duration /= 100 

avgScores = np.mean(cvScores_duration)
stdScores = np.std(cvScores_duration)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
 
baseline = np.mean(y_train_full)
mae_baseline = np.mean(np.abs(y_train_full - baseline))

# full training results (test data not seen during cross val)
y_pred_train_dur = model_duration.predict(X_train_full) 

# Predict on test data
y_pred_test_dur = model_duration.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full, y_pred_train)   
mae_test_dur= mean_absolute_error(y_test, y_pred_test)  
  
# Print accuracy and F1 score
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)
 

Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y_contractions = np.concatenate([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])       

y_durations = np.concatenate([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])       

y = np.column_stack((y_contractions, y_durations))

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X_zygo)[2]
epoch_len = np.shape(X_zygo)[1]

In [ ]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 

cvScores_dur =[]
cvScores_contr =[]
cvScores = [] 

epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_twohead = CNN_model_twohead(input_shape, num_classes,feature_num) #CNN_model_regression if treating both as continuous
    model_twohead.compile(optimizer='adam',loss=['sparse_categorical_crossentropy', 'mae'],metrics=['accuracy', 'mae'] ) #think about metric 
    model_history_kfold = model_twohead.fit(X_train,[y_train[:,0], y_train[:,1]], 
                                            validation_data=(X_val,[y_val[:,0], y_val[:,1]]), 
                                            epochs=epoch_num)
    
    scores = model_twohead.evaluate(X_test,[y_test[:,0], y_test[:,1]])
    metrics = dict(zip(model_twohead.metrics_names, scores))

    cvScores_dur.append(metrics['duration_output_mae'])
    cvScores_contr.append(metrics['count_output_accuracy'] * 100)


    cvScores.append(scores)
 
    

  
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    #model_history_kfold = model.fit(X_train, y_train, epochs=epoch_num, validation_data=(X_val, y_val)) #, callbacks=[early_stop])
    
    #plot_accuracy(model_history_kfold,i)
    
    

    k += 1 
    

model_history_twohead = model_twohead.fit(X_train_full, [y_train_full[:,0], y_train_full[:,1]], epochs=epoch_num, validation_data=(X_test, [y_test[:,0], y_test[:,1]])) #, callbacks=[early_stop])

In [ ]:
cvScores

In [ ]:
# comparison for two head model 
# cross validation results 
avgScores_dur = np.mean(cvScores,axis=0)[3]
stdScores_dur = np.std(cvScores,axis=0)[3]

avgScores_contr= np.mean(cvScores,axis=0)[6]
stdScores_contr = np.std(cvScores,axis=0)[6]
 

print(f"Average KFold Cross Validation Score for contraction: {avgScores_contr}")
print(f"Standard Deviation KFold Cross Validation Score for contractionon: {stdScores_contr}")

print(f"Average KFold Cross Validation Score for duration: {avgScores_dur}")
print(f"Standard Deviation KFold Cross Validation Score for duration: {avgScores_dur}")

# full training results (test data not seen during cross val)
[y_pred_train_contraction, y_pred_train_dur] = model_twohead.predict(X_train_full)  

y_pred_train_contraction = np.argmax(y_pred_train_contraction, axis=1)   

# Predict on test data
[y_pred_test_contraction, y_pred_test_dur] = model_twohead.predict(X_test)  

y_pred_test_contraction = np.argmax(y_pred_test_contraction, axis=1)   

# Calculate accuracy
accuracy_training_contraction = accuracy_score(y_train_full[:,0], y_pred_train_contraction)   
accuracy_test_contraction = accuracy_score(y_test[:,0], y_pred_test_contraction)  

# Calculate F1 score
f1_training_contraction = f1_score(y_train_full[:,0], y_pred_train_contraction, average='weighted')  
f1_test_contraction = f1_score(y_test[:,0], y_pred_test_contraction, average='weighted')  

# MAE 
baseline = np.mean(y_train_full[:,1])
mae_baseline = np.mean(np.abs(y_train_full[:,1] - baseline))

y_pred_train_dur = model_twohead.predict(X_train_full) 
y_pred_test_dur = model_twohead.predict(X_test) 

  
mae_training_dur = mean_absolute_error(y_train_full[:,1], y_pred_train)   
mae_test_dur= mean_absolute_error(y_test[:,1], y_pred_test)  
  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training_contraction)  
print("Test Accuracy :", accuracy_test_contraction)  
print("Training F1 Score :", f1_training_contraction)   
print("Test F1 Score :", f1_test_contraction)   
print("----------------------") 
print("Baseline MAE:", mae_baseline)
print("Training MAE :", mae_training_dur)
print("Test MAE :", mae_test_dur)


## Two Channel 

Single Head Count 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]
 

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model_contraction_multichan(input_shape, num_classes,feature_num)
    model.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'sparse_categorical_crossentropy',
        'corr_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'zygo_output': ['accuracy'],
        'corr_output': ['accuracy']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]])) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores.append(scores[1] * 100)

    k += 1 
    

model_history = model.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) 

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model.predict(X_test)

# Convert probabilities to class labels
y_pred_train_zygo = np.argmax(y_pred_train_zygo, axis=1)
y_pred_train_corr = np.argmax(y_pred_train_corr, axis=1)

y_pred_test_zygo = np.argmax(y_pred_test_zygo, axis=1)
y_pred_test_corr = np.argmax(y_pred_test_corr, axis=1)

# true labels for corr and zygo 
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]
y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# calculate accuracy for each muscle group 
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr)

# calculate f1 score for each muscle group 
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr, average='weighted')

print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n -------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)

# average score
print("\n -------- Average --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)  

Single Head Duration 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])




indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_dur = CNN_model_duration_multichan(input_shape, num_classes,feature_num)
    model_dur.compile(
    optimizer='adam',
    loss={
        'zygo_output': 'mae',
        'corr_output': 'mae'
    },
    metrics={
        'zygo_output': ['mae'],
        'corr_output': ['mae']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_dur.fit(X_train, [y_train[:, 0], y_train[:, 1]], epochs=epoch_num, validation_data=(X_val, [y_val[:, 0], y_val[:, 1]])) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)
    
    scores = model_dur.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores.append(scores)

    k += 1 
    

model_history = model_dur.fit(X_train_full, [y_train_full[:, 0],y_train_full[:, 1]], epochs=epoch_num, validation_data=(X_test, [y_test[:, 0],y_test[:, 1]])) 

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores,axis=0)[1]
stdScores = np.std(cvScores,axis=0)[1]

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")
# full training results (test data not seen during cross val)
y_pred_train_zygo, y_pred_train_corr = model_dur.predict(X_train_full)
y_pred_test_zygo, y_pred_test_corr = model_dur.predict(X_test)


baseline_zygo = np.mean(y_train_full[:,0])
mae_baseline_zygo = np.mean(np.abs(y_train_full[:,0] - baseline_zygo))

baseline_corr = np.mean(y_train_full[:,1])
mae_baseline_corr = np.mean(np.abs(y_train_full[:,1] - baseline_corr))
  
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,0], y_pred_train_zygo)   
mae_test_dur_zygo = mean_absolute_error(y_test[:,0], y_pred_test_zygo)  

mae_training_dur_corr = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo)   
mae_test_dur_corr= mean_absolute_error(y_test[:,1], y_pred_test_zygo)  
# Print accuracy and F1 score

# Print accuracy and F1 score
print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)
print("----------------------") 
print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)

Two Head 

In [ ]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    features_all_temp = features_all.loc[features_all["Subject"] != subj_omit]

else:
    features_all_temp = features_all


X_zygo = features_all_temp[["Zygo"]] 
X_corr = features_all_temp[["Corr"]]

X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=2) # feeding in two channels 


y_contr = np.transpose([features_all_temp["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all_temp["Num_Contractions_Corr"].astype(int).to_numpy()])


y_dur = np.transpose([features_all_temp["Duration_zygo"],
                    features_all_temp["Duration_corr"]])


y = np.column_stack((y_contr, y_dur))
y = y[:, [0, 2, 1, 3]]

indices = np.arange(len(X))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42) # keep 20% purely for testing 

# split like this to be able to recover the indices for later when rescoring 
X_train_full = X[idx_train]
X_test = X[idx_test]
y_train_full = y[idx_train]
y_test = y[idx_test]

input_shape = X_train_full.shape[1:]   
num_classes = len(np.unique(y))    

input_shape = (input_shape[0], 1)   
feature_num = np.shape(X)[2]
epoch_len = np.shape(X)[1]

In [ ]:
np.shape(y)

In [86]:
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores_fourhead=[]
epoch_num = 10 
k = 1

for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {k} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model_fourhead = CNN_model_fourhead_multichan(input_shape, num_classes,feature_num)
    model_fourhead.compile(
    optimizer='adam',
    loss={
        'zygo_count_output': 'sparse_categorical_crossentropy',
        'corr_count_output': 'sparse_categorical_crossentropy',
        'zygo_duration_output': 'mae',
        'corr_duration_output': 'mse'
    },
    metrics={
        'zygo_count_output': ['accuracy'],
        'corr_count_output': ['accuracy'],
        'zygo_duration_output': ['mae'],
        'corr_duration_output': ['mae']
    }
)
    #early_stop = EarlyStopping(monitor='val_loss',  patience=3, restore_best_weights=True)
    
    model_history_kfold = model_fourhead.fit(X_train, {
        'zygo_count_output': y_train[:, 0],
        'zygo_duration_output': y_train[:, 1],
        'corr_count_output': y_train[:, 2],
        'corr_duration_output': y_train[:, 3]
    }, epochs=epoch_num, validation_data=(
    X_val,
    {
        'zygo_count_output': y_val[:, 0],
        'zygo_duration_output': y_val[:, 1],
        'corr_count_output': y_val[:, 2],
        'corr_duration_output': y_val[:, 3]
    }
)) #, callbacks=[early_stop])
    #plot_accuracy(model_history_kfold,i)

    
    scores = model_fourhead.evaluate(X_test,[y_test[:, 0], y_test[:, 1]])
    print(scores)
    cvScores_fourhead.append(scores)

    k += 1 
    


Fold: 1 ==================================================================
Epoch 1/10
143/143 [==============================] - 14s 81ms/step - loss: 22403.8945 - zygo_count_output_loss: 8.6866 - zygo_duration_output_loss: 52.5360 - corr_count_output_loss: 17.9809 - corr_duration_output_loss: 22324.6816 - zygo_count_output_accuracy: 0.4202 - zygo_duration_output_mae: 52.5360 - corr_count_output_accuracy: 0.3686 - corr_duration_output_mae: 71.9715 - val_loss: 95.6145 - val_zygo_count_output_loss: 1.4339 - val_zygo_duration_output_loss: 1.8772 - val_corr_count_output_loss: 3.8920 - val_corr_duration_output_loss: 88.4114 - val_zygo_count_output_accuracy: 0.3325 - val_zygo_duration_output_mae: 1.8772 - val_corr_count_output_accuracy: 0.7568 - val_corr_duration_output_mae: 5.4749
Epoch 2/10
143/143 [==============================] - 9s 64ms/step - loss: 5556.7788 - zygo_count_output_loss: 1.9002 - zygo_duration_output_loss: 24.8244 - corr_count_output_loss: 6.1323 - corr_duration_output_lo

In [87]:

# Cross-validation results
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")


y_pred_train = model_fourhead.predict(X_train_full)
y_pred_test = model_fourhead.predict(X_test)

y_pred_train_zygo_count, y_pred_train_zygo_dur, y_pred_train_corr_count, y_pred_train_corr_dur = y_pred_train
y_pred_test_zygo_count, y_pred_test_zygo_dur, y_pred_test_corr_count, y_pred_test_corr_dur = y_pred_test


# baselines
baseline_zygo = np.mean(y_train_full[:,1])
baseline_corr = np.mean(y_train_full[:,3])

mae_baseline_zygo = np.mean(np.abs(y_train_full[:,1] - baseline_zygo))
mae_baseline_corr = np.mean(np.abs(y_train_full[:,3] - baseline_corr))

# MAE
mae_training_dur_zygo = mean_absolute_error(y_train_full[:,1], y_pred_train_zygo_dur)
mae_test_dur_zygo = mean_absolute_error(y_test[:,1], y_pred_test_zygo_dur)

mae_training_dur_corr = mean_absolute_error(y_train_full[:,3], y_pred_train_corr_dur)
mae_test_dur_corr = mean_absolute_error(y_test[:,3], y_pred_test_corr_dur)

print("Baseline MAE corr:", mae_baseline_corr)
print("Training MAE corr :", mae_training_dur_corr)
print("Test MAE corr:", mae_test_dur_corr)

print("----------------------") 

print("Baseline MAE zygo:", mae_baseline_zygo)
print("Training MAE zygo :", mae_training_dur_zygo)
print("Test MAE zygo:", mae_test_dur_zygo)



y_pred_train_zygo_count = np.argmax(y_pred_train_zygo_count, axis=1)
y_pred_train_corr_count = np.argmax(y_pred_train_corr_count, axis=1)

y_pred_test_zygo_count = np.argmax(y_pred_test_zygo_count, axis=1)
y_pred_test_corr_count = np.argmax(y_pred_test_corr_count, axis=1)

# true labels
y_train_zygo = y_train_full[:, 0]
y_train_corr = y_train_full[:, 1]

y_test_zygo = y_test[:, 0]
y_test_corr = y_test[:, 1]

# accuracy
accuracy_train_zygo = accuracy_score(y_train_zygo, y_pred_train_zygo_count)
accuracy_train_corr = accuracy_score(y_train_corr, y_pred_train_corr_count)

accuracy_test_zygo = accuracy_score(y_test_zygo, y_pred_test_zygo_count)
accuracy_test_corr = accuracy_score(y_test_corr, y_pred_test_corr_count)

# f1
f1_train_zygo = f1_score(y_train_zygo, y_pred_train_zygo_count, average='weighted')
f1_train_corr = f1_score(y_train_corr, y_pred_train_corr_count, average='weighted')

f1_test_zygo = f1_score(y_test_zygo, y_pred_test_zygo_count, average='weighted')
f1_test_corr = f1_score(y_test_corr, y_pred_test_corr_count, average='weighted')



print("-------- Zygo --------")
print("Training Accuracy:", accuracy_train_zygo)
print("Test Accuracy:", accuracy_test_zygo)
print("Training F1 Score:", f1_train_zygo)
print("Test F1 Score:", f1_test_zygo)

print("\n-------- Corr --------")
print("Training Accuracy:", accuracy_train_corr)
print("Test Accuracy:", accuracy_test_corr)
print("Training F1 Score:", f1_train_corr)
print("Test F1 Score:", f1_test_corr)


print("\n-------- Average (Counts) --------")
print("Training Accuracy:", (accuracy_train_zygo + accuracy_train_corr) / 2)
print("Test Accuracy:", (accuracy_test_zygo + accuracy_test_corr) / 2)
print("Training F1 Score:", (f1_train_zygo + f1_train_corr) / 2)
print("Test F1 Score:", (f1_test_zygo + f1_test_corr) / 2)

print("\n-------- Average (Duration MAE) --------")
print("Training MAE:", (mae_training_dur_zygo + mae_training_dur_corr) / 2)
print("Test MAE:", (mae_test_dur_zygo + mae_test_dur_corr) / 2)

Average KFold Cross Validation Score: 8.226582050323486
Standard Deviation KFold Cross Validation Score: 3.631018823393753
45/45 [==============================] - 1s 11ms/step
Baseline MAE corr: 5.768655971313989
Training MAE corr : 4.9962482616491295
Test MAE corr: 5.202696081222697
----------------------
Baseline MAE zygo: 1.1610909069510156
Training MAE zygo : 1.0717061739988856
Test MAE zygo: 1.0583033799311192
-------- Zygo --------
Training Accuracy: 0.7233893557422969
Test Accuracy: 0.7247899159663865
Training F1 Score: 0.7118264808446533
Test F1 Score: 0.7139333706559452

-------- Corr --------
Training Accuracy: 0.726890756302521
Test Accuracy: 0.7156862745098039
Training F1 Score: 0.6119323641865505
Test F1 Score: 0.5970868347338936

-------- Average (Counts) --------
Training Accuracy: 0.7251400560224089
Test Accuracy: 0.7202380952380952
Training F1 Score: 0.661879422515602
Test F1 Score: 0.6555101026949194

-------- Average (Duration MAE) --------
Training MAE: 3.033977217

## Segmentation

In [ ]:
def comparator(learner, instructor):
    if len(learner) != len(instructor):
        raise AssertionError("Layer count mismatch")
    for a, b in zip(learner, instructor):
        if tuple(a) != tuple(b):
            print(colored("Test failed", attrs=['bold']))
            raise AssertionError("Error in test")
    print(colored("All tests passed!", "green"))

def summary(model):
    result = []
    for layer in model.layers:
        output_shape = getattr(layer.output, 'shape', None)
        params = layer.count_params() if hasattr(layer, 'count_params') else 0
        result.append([layer.__class__.__name__, output_shape, params])
    return result
